In [1]:
import pandas as pd
import numpy as np
from datetime import date, timedelta
import pyodbc as db

In [3]:
### Load the Customer Demo dataframe
Customer_demo_file = "Customer_demo_information.csv"
Customer_demo_df = pd.read_csv (Customer_demo_file)
Customer_demo_df = pd.DataFrame(Customer_demo_df)
# Check and see the df
Customer_demo_df.count()
### Load the Customer Address dataframe
Customer_address_file = "Cust_address.csv"
Customer_address_df = pd.read_csv(Customer_address_file)
Customer_address_df = pd.DataFrame(Customer_address_df)
# Check and see the df
Customer_address_df.count()
# organize customer table
## Clean up the dataset
# Replace 'F' with 'Female' in the 'gender' column
Customer_demo_df.loc[Customer_demo_df['gender'] == 'F', 'gender'] = 'Female'
Customer_demo_df.loc[Customer_demo_df['gender'] == 'Femal', 'gender'] = 'Female'
Customer_demo_df.loc[Customer_demo_df['gender'] == 'M', 'gender'] = 'Male'
Customer_demo_df.loc[Customer_demo_df['gender'] == 'U', 'gender'] = 'NAs'
print(Customer_demo_df['gender'].unique())

# Drop not needed columns
Customer_demo_df = Customer_demo_df.drop(columns=['default'])
print(Customer_demo_df)

print(Customer_demo_df['deceased_indicator'].unique())
#print(Customer_address_df)
# change country and state to US 
print(Customer_address_df['state'].unique())
print(Customer_address_df['country'].unique())


Customer_address_df['country'].replace(['Australia'], 'United States', inplace=True)

Customer_address_df['state'].replace(['New South Wales'], 'Massachusetts', inplace=True)
Customer_address_df['state'].replace(['QLD'], 'Maine', inplace=True)
Customer_address_df['state'].replace(['VIC'], 'California', inplace=True)
Customer_address_df['state'].replace(['NSW'], 'New Hampshire', inplace=True)
Customer_address_df['state'].replace(['Victoria'], 'New York', inplace=True)

# Joining all the customer table 
CustomerDim = Customer_address_df.merge(Customer_demo_df, on='customer_id')
CustomerDim = CustomerDim.drop(columns=['postcode', 'property_valuation', 'past_3_years_bike_related_purchases', 'DOB', 'job_title',
                                       'job_industry_category', 'wealth_segment', 'tenure'])
CustomerDim['CustomerDim_ID'] = None
CustomerDim['CustomerDim_ID']= CustomerDim.index + 1
# Delta and loading the delta:
CustomerDim_delta = CustomerDim[55:75]

['Female' 'Male' 'NAs' nan]
      customer_id      first_name  last_name  gender  \
0             1.0         Laraine  Medendorp  Female   
1             2.0             Eli    Bockman    Male   
2             3.0           Arlin     Dearle    Male   
3             4.0          Talbot        NaN    Male   
4             5.0  Sheila-kathryn     Calton  Female   
...           ...             ...        ...     ...   
3996       3997.0          Blanch     Nisuis  Female   
3997       3998.0          Sarene    Woolley     NAs   
3998       3999.0       Patrizius        NaN    Male   
3999       4000.0           Kippy    Oldland    Male   
4000          NaN             NaN        NaN     NaN   

      past_3_years_bike_related_purchases         DOB               job_title  \
0                                    93.0  1953-10-12     Executive Secretary   
1                                    81.0  1980-12-16  Administrative Officer   
2                                    61.0  1954-01-20   

In [4]:
### Load the Sales dataframe
Accessory_Salesfile = "Sales_csv.csv"
Sales_df = pd.read_csv (Accessory_Salesfile)
Sales_df = pd.DataFrame(Sales_df)
# Check and see the df
Sales_df.count()
##### Data cleanup: First merge two sales df together######################### Sales.csv
# First clean up both of them
#display(Accessory_Sales_df.filter(['Date']))

### Covert the date to 2023 to match the bike trip data for future analysis. 
# Convert the 'date_column' to datetime format
Sales_df['date_column'] = pd.to_datetime(Sales_df['Date'])

# Function to update the year to 2023
def update_year(date):
    return date.replace(year=2023)

# Apply the function to the 'date_column'
Sales_df['date_column'] = Sales_df['date_column'].apply(update_year)

# Drop not needed columns
Sales_df = Sales_df.rename(columns={'date_column': 'EffectiveDate', 'Sales Cost': 'Sales_Price'})
Sales_df = Sales_df.rename(columns={'Product Category': 'Product_Category', 'Sub Category' :'Sub_Category'})
Sales_df = Sales_df.drop(columns=['Hour', 'Date', 'Hour', 'Year', 'Time', 'Customer Age', 'Customer Gender', 'Payment', 'Month', 'Unit Cost', 'Country', 'State'])
# Adding columns
Sales_df.loc[:, 'Product_ID'] = None
Sales_df.loc[:, 'CurrentFlag'] = None
Sales_df.loc[:, 'ExpireDate'] = None
product_id = range(100)
Sales_df['Product_ID'] = np.random.choice(product_id, size=len(Sales_df))
Sales_df['CurrentFlag'] = 'Current'
Sales_df['ExpireDate'] = pd.to_datetime('2099-12-31')
#### NOW seperate product intp bike and bike accessories
## Preparing for the dimension table
display(Sales_df['Product_Category'].unique())

# Create Accessory dim
Accessory_df = Sales_df[Sales_df['Product_Category'].isin(['Accessories', 'Clothing'])].copy()

# Create Bike dim
BikeProduct_df = Sales_df[Sales_df['Product_Category'] == 'Bikes'].copy()
# Bike Product Dim
# Adding Bike Product Dimension ID
BikeProduct_df.loc[:, 'BikesDim_ID'] = 1  # Initialize to 1
BikeProduct_df['BikesDim_ID'] += np.arange(1, len(BikeProduct_df) + 1)  # Increment by 1 for each row
Bike_Delta = BikeProduct_df[10:80]
print(Bike_Delta)

array(['Accessories', 'Clothing', 'Bikes'], dtype=object)

     Unnamed: 0 Product_Category    Sub_Category  Quantity  Revenue  Rating  \
73           79            Bikes  Mountain Bikes         1      873     8.0   
74           80            Bikes   Touring Bikes         3      751     8.0   
75           81            Bikes  Mountain Bikes         1     2340     6.5   
76           82            Bikes  Mountain Bikes         2      707     7.5   
77           83            Bikes  Mountain Bikes         1     2340     9.0   
..          ...              ...             ...       ...      ...     ...   
346         358            Bikes      Road Bikes         2     2251     5.0   
347         359            Bikes  Mountain Bikes         1     2376     8.0   
348         360            Bikes   Touring Bikes         3      770     9.5   
349         361            Bikes      Road Bikes         1     2318     7.0   
350         362            Bikes   Touring Bikes         3      659     9.0   

     Sales_Price EffectiveDate  Product_ID CurrentF

In [5]:
### Load the Shops dataframe
Shop_file = "Walmart.csv"
Shop_df = pd.read_csv(Shop_file)
Shop_df = pd.DataFrame(Shop_df)
Shop_df.count()
print(Shop_df)
# Drop not needed columns
updated_Shop_df = Shop_df.drop(columns=['Holiday_Flag', 'Unemployment', 'CPI', 'Weekly_Sales', 'Date', 'Store'])
# add add new columns
updated_Shop_df.loc[:, 'State'] = None
updated_Shop_df.loc[:, 'Country'] = None
updated_Shop_df.loc[:, 'Product_ID'] = None
updated_Shop_df.loc[:, 'Store_ID'] = None
# Populate the columns
state_names = ['Massachusetts', 'Maine', 'California', 'New Hampshire', 'New York']
country_name = ['United States']

# Generate a DataFrame with 3217 records and populate the 'State' and 'Country' columns
updated_Shop_df['State'] = np.random.choice(state_names, size=len(updated_Shop_df))
updated_Shop_df['Country'] = np.random.choice(country_name, size=len(updated_Shop_df))

# product id
random_product_ids = np.random.randint(1, 100, size=len(updated_Shop_df))
# Add the 'Product_ID' column to the DataFrame
updated_Shop_df['Product_ID'] = random_product_ids

# Store id
store_id = range(100)
updated_Shop_df['Store_ID'] = np.random.choice(store_id, size=len(updated_Shop_df))
Bikeshopdf = updated_Shop_df[0:3217]
Bikeshopdf['BikeShop_Dim_ID'] = Bikeshopdf.index + 1
Bikeshopdf_delta = Bikeshopdf[150:200]

      Store        Date  Weekly_Sales  Holiday_Flag  Temperature  Fuel_Price  \
0         1  05-02-2010    1643690.90             0        42.31       2.572   
1         1  12-02-2010    1641957.44             1        38.51       2.548   
2         1  19-02-2010    1611968.17             0        39.93       2.514   
3         1  26-02-2010    1409727.59             0        46.63       2.561   
4         1  05-03-2010    1554806.68             0        46.50       2.625   
...     ...         ...           ...           ...          ...         ...   
6430     45  28-09-2012     713173.95             0        64.88       3.997   
6431     45  05-10-2012     733455.07             0        64.89       3.985   
6432     45  12-10-2012     734464.36             0        54.47       4.000   
6433     45  19-10-2012     718125.53             0        56.47       3.969   
6434     45  26-10-2012     760281.43             0        58.85       3.882   

             CPI  Unemployment  
0     

C:\Users\chloe\AppData\Local\Temp\ipykernel_11176\2321014280.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Bikeshopdf['BikeShop_Dim_ID'] = Bikeshopdf.index + 1


In [6]:
### Load the Customer and Transaction dataframe
Transaction_file = "Transaction.csv"
Transaction_df = pd.read_csv(Transaction_file)
Transaction_df = pd.DataFrame(Transaction_df)
Transaction_df.count()
print(Transaction_df)
# Convert the 'date_column' to datetime format
Transaction_df['transaction_date'] = pd.to_datetime(Transaction_df['transaction_date'])

# Function to update the year to 2023
def update_year(date):
    return date.replace(year=2023)

# Apply the function to the 'date_column'
Transaction_df['new_transaction_date'] = Transaction_df['transaction_date'].apply(update_year)

Transaction_df = Transaction_df.drop(columns=['transaction_date', 'product_line', 'product_class', 'product_size', 'customer_id', 'standard_cost'])
# add add new columns
Transaction_df.loc[:, 'Store_ID'] = None
Transaction_df.loc[:, 'Customer_ID'] = None
Transaction_df.loc[:, 'Transaction_2023_first_quarter'] = None
Transaction_df.loc[:, 'Transaction_2023_sec_quarter'] = None
Transaction_df.loc[:, 'Transaction_2023_third_quarter'] = None
Transaction_df.loc[:, 'Transaction_2023_fourth_quarter'] = None
# Store id
store_id = range(100)
Transaction_df['Store_ID'] = np.random.choice(store_id, size=len(Transaction_df))

# Customer id
customer_id = range(100)
Transaction_df['Customer_ID'] = np.random.choice(customer_id, size=len(Transaction_df))
Transaction_Delta = Transaction_df[100:200]

       transaction_id  product_id  customer_id transaction_date online_order  \
0                   1           2         2950        2/25/2017        False   
1                   2           3         3120        5/21/2017         True   
2                   3          37          402       10/16/2017        False   
3                   4          88         3135        8/31/2017        False   
4                   5          78          787        10/1/2017         True   
...               ...         ...          ...              ...          ...   
19995           19996          51         1018        6/24/2017         True   
19996           19997          41          127        11/9/2017         True   
19997           19998          87         2284        4/14/2017         True   
19998           19999           6         2764         7/3/2017        False   
19999           20000          11         1144        9/22/2017         True   

      order_status           brand prod

In [7]:
# Load the bike trips data, since all the trips data are from 2023, so I will be name each df after the month
# Since the dataset have too many observationsm I will only read 2000 rows from each of them. 
num_rows_to_read = 2000

TripJanfile = "202301tripdata.csv"
TripJan_df = pd.read_csv(TripJanfile, nrows=num_rows_to_read)
TripJan_df = pd.DataFrame(TripJan_df)


TripFebfile = "202302tripdata.csv"
TripFeb_df = pd.read_csv(TripFebfile, nrows=num_rows_to_read)
TripJFeb_df = pd.DataFrame(TripFeb_df)

TripMarfile = "202303tripdata.csv"
TripMar_df = pd.read_csv(TripMarfile, nrows=num_rows_to_read)
TripMar_df = pd.DataFrame(TripMar_df)

TripAprilfile = "202304tripdata.csv"
TripApril_df = pd.read_csv(TripAprilfile, nrows=num_rows_to_read)
TripApril_df = pd.DataFrame(TripApril_df)
# Merging all the dfs
bike_station = pd.merge(TripJan_df, TripFeb_df, how='outer')
bike_station = pd.merge(bike_station, TripMar_df, how='outer')
bike_station = pd.merge(bike_station, TripApril_df, how='outer')

# Drop not needed columns
Bike_sharing_dim = bike_station.drop(columns=['rideable_type','start_station_name', 'end_station_name', 'start_lat', 'start_lng',
                                             'end_lat', 'end_lng', 'ended_at'])
# Create New columns
Bike_sharing_dim['State'] = None
Bike_sharing_dim['Country'] = None
Bike_sharing_dim['Customer_ID'] = None
Bike_sharing_dim['BikeSharingDim_ID'] = None
Bike_sharing_dim['Member_ship_effective_date'] = None
# Generate random dates for 'member' entries
member_indices = Bike_sharing_dim['member_casual'] == "member"
num_members = member_indices.sum()

random_dates = pd.to_datetime(
    np.random.choice(pd.date_range('2022-01-01', '2022-12-31'), size=num_members),
    errors='coerce')

# Assign random dates to 'Member_ship_effective_date'
Bike_sharing_dim.loc[member_indices, 'Member_ship_effective_date'] = random_dates
# Define the list of state names and country name
state_names = ['Massachusetts', 'Maine', 'California', 'New Hampshire', 'New York']
country_name = ['United States']

# Generate a DataFrame with 1066014 records and populate the 'State' and 'Country' columns
Bike_sharing_dim['State'] = np.random.choice(state_names, size=len(Bike_sharing_dim))
Bike_sharing_dim['Country'] = np.random.choice(country_name, size=len(Bike_sharing_dim))


# Adding Customer_id
random_customer_ids = np.random.randint(1, 100, size=len(Bike_sharing_dim))

# Add the 'Product_ID' column to the DataFrame
Bike_sharing_dim['Customer_ID'] = random_customer_ids


# adding index based id
Bike_sharing_dim['BikeSharingDim_ID']= Bike_sharing_dim.index + 1
#Creat and load the delta
Bike_sharing_dim_delta = Bike_sharing_dim[1:500]

In [8]:
############# Create Fact 
BikeSaleFact_df = pd.DataFrame({
    'CustomerDim_ID': CustomerDim_delta['CustomerDim_ID'],
    'BikesDim_ID': Bike_Delta['BikesDim_ID'],
    'BikeShop_Dim_ID': Bikeshopdf_delta['BikeShop_Dim_ID'],
    'transaction_id': Transaction_Delta['transaction_id'],
    'T_transaction_date': Transaction_Delta['new_transaction_date'],
    'BikeSharingDim_ID': Bike_sharing_dim_delta['BikeSharingDim_ID']
})

BikeSaleFact_df['T_transaction_date'] = BikeSaleFact_df['T_transaction_date'].dt.strftime('%Y-%m-%d %H:%M:%S')

BikeSaleFact_df_sml = BikeSaleFact_df[270:275]
print(BikeSaleFact_df_sml)

     CustomerDim_ID  BikesDim_ID  BikeShop_Dim_ID  transaction_id  \
271             NaN          NaN              NaN             NaN   
272             NaN          NaN              NaN             NaN   
273             NaN          NaN              NaN             NaN   
274             NaN          NaN              NaN             NaN   
275             NaN          NaN              NaN             NaN   

    T_transaction_date  BikeSharingDim_ID  
271                NaN                272  
272                NaN                273  
273                NaN                274  
274                NaN                275  
275                NaN                276  


In [9]:
############# Create Fact 
BikeSaleFact_df = pd.DataFrame({
    'CustomerDim_ID': CustomerDim_delta['CustomerDim_ID'],
    'BikesDim_ID': Bike_Delta['BikesDim_ID'],
    'BikeShop_Dim_ID': Bikeshopdf_delta['BikeShop_Dim_ID'],
    'transaction_id': Transaction_Delta['transaction_id'],
    'T_transaction_date': Transaction_Delta['new_transaction_date'],
    'BikeSharingDim_ID': Bike_sharing_dim_delta['BikeSharingDim_ID']
})

BikeSaleFact_df['T_transaction_date'] = BikeSaleFact_df['T_transaction_date'].dt.strftime('%Y-%m-%d %H:%M:%S')

###########################################################################################
# Load the Bike_trip_Dim Delta data
import pyodbc as db
conn = db.connect('Driver={your_driver};'
                 'SERVER=your_server;'
                'SERVER=your_server;'
                'Database=Bike;'
                 'Trusted_Connection=yes;')

cursor = conn.cursor()

for index, row in BikeSaleFact_df_sml.iterrows():
    cursor.execute("INSERT into BikeSaleFact (CurrentCustDim_ID, Product_BikeDim_ID, Bike_ShopDim_ID, TransactionDim_ID, BikeSharingDim_ID, T_transaction_date) values (?, ?, ?, ?, ?, ?)",
                   row.CustomerDim_ID, row.BikesDim_ID, row.BikeShop_Dim_ID, row.transaction_id, row.BikeSharingDim_ID, row.T_transaction_date)
    
    
conn.commit()
cursor.close()


ProgrammingError: ('42000', '[42000] [Microsoft][ODBC SQL Server Driver][SQL Server]The incoming tabular data stream (TDS) remote procedure call (RPC) protocol stream is incorrect. Parameter 4 (""): The supplied value is not a valid instance of data type float. Check the source data for invalid values. An example of an invalid value is data of numeric type with scale greater than precision. (8023) (SQLExecDirectW)')